# TensorRT 기초: PyTorch → ONNX → TensorRT

> 실행 환경: Colab GPU 런타임 (VS Code: **Select Kernel → Colab → GPU**)
> study와 무관한 boilerplate(캐시, 추론 래퍼, 벤치마크)는 [`src/vopt`](../src/vopt)에 있습니다.

## 0. Setup
- Colab에 기본 설치된 torch는 그대로 사용하고, 없는 패키지만 `uv`로 설치합니다.
- 이 저장소를 clone해서 `vopt` 헬퍼를 import합니다. 로컬에서 `src/vopt`를 수정했다면 push한 뒤 이 셀을 다시 실행하세요(`git pull` + autoreload).

In [6]:
# 0. Setup (Colab 런타임에서 실행)
import os, subprocess, sys

REPO_URL = "https://github.com/limJhyeok/vopt-learning.git"
REPO_DIR = "/content/vopt-learning"
PKGS = ["tensorrt-cu12", "onnx",]  # Colab 기본 torch/torchvision은 재설치하지 않음

assert os.path.isdir("/content"), "Colab 런타임이 아닙니다. 커널을 Colab으로 선택하세요."

def sh(cmd: str) -> None:
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

sh("pip install -q uv")
if os.path.isdir(REPO_DIR):
    sh(f"git -C {REPO_DIR} pull -q --ff-only")
else:
    sh(f"git clone -q --depth 1 {REPO_URL} {REPO_DIR}")
sh("uv pip install --system -q " + " ".join(PKGS))

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# autoreload: src/vopt 수정 사항을 커널 재시작 없이 반영
# Colab의 구버전 IPython autoreload는 Python 3.12+에서 삭제된 `imp`를 import하므로 최소한의 shim을 넣는다.
import importlib, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401  (Python <= 3.11)
    except ModuleNotFoundError:
        sys.modules["imp"] = types.SimpleNamespace(reload=importlib.reload)
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as e:
    print(f"[warn] autoreload 비활성화 ({e!r}). vopt 수정 후에는 커널을 재시작하세요.")

from vopt.env import print_env
print_env()

$ pip install -q uv
$ git -C /content/vopt-learning pull -q --ff-only
$ uv pip install --system -q tensorrt-cu12 onnx
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
python       : 3.13.16
torch        : 2.14.1+cu130
cuda (torch) : 13.0
gpu          : Tesla T4
gpu tag      : tesla-t4-sm75
tensorrt     : 11.4.0.106


> **(선택) Google Drive 캐시**: 명령 팔레트 → `Colab: Mount Google Drive to Server...`로 Drive를 마운트하면,
> 빌드한 engine이 `MyDrive/vopt-cache`에 저장되어 런타임이 바뀌어도 유지됩니다. (마운트하지 않으면 `/content/vopt-cache`)

## 1. PyTorch 모델 → ONNX

In [7]:
import copy
import torch
import torchvision
from vopt import cache, data

B = 8
model = torchvision.models.resnet18(weights="DEFAULT").eval().cuda()
# 실제 사진 B장(Oxford-IIIT Pet, CC BY-SA 4.0. HF에서 필요한 장수만 받음)을 ResNet 표준 전처리한 [B, 3, 224, 224] 배치.
# 랜덤 노이즈는 logit 1·2위 차이가 작아 정밀도 비교(top-1 일치율)가 부정확하다.
x = data.imagenet_batch(B)

# 최신 TensorRT는 strongly typed: engine 정밀도 = ONNX의 dtype.
# ONNX/engine 입력 shape은 export할 때의 x shape으로 고정되므로 파일명에 B를 넣는다.
onnx_path = cache.cache_dir() / f"resnet18_b{B}.onnx"
if not onnx_path.exists():
    # dynamo=False: 기존 TorchScript 기반 exporter. 에러가 나면 PKGS에 "onnxscript"를 추가하고 dynamo=True로 바꾸세요.
    torch.onnx.export(
        model, (x,), str(onnx_path),
        input_names=["input"], output_names=["logits"],
        opset_version=17, dynamo=False,
    )
print(onnx_path, f"{onnx_path.stat().st_size / 1e6:.1f} MB")

# → FP16 engine을 원하면 FP16 모델 자체를 export한다.
model_fp16 = copy.deepcopy(model).half()
x_fp16 = x.half()

fp16_onnx_path = cache.cache_dir() / f"resnet18_fp16_b{B}.onnx"
if not fp16_onnx_path.exists():
    # dynamo=False: 기존 TorchScript 기반 exporter. 에러가 나면 PKGS에 "onnxscript"를 추가하고 dynamo=True로 바꾸세요.
    torch.onnx.export(
        model_fp16, (x_fp16,), str(fp16_onnx_path),
        input_names=["input"], output_names=["logits"],
        opset_version=17, dynamo=False,
    )
print(fp16_onnx_path, f"{fp16_onnx_path.stat().st_size / 1e6:.1f} MB")

/tmp/ipykernel_8481/2958115840.py:14: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


/content/vopt-cache/resnet18.onnx 46.7 MB
/content/vopt-cache/resnet18_fp16.onnx 23.4 MB


## 2. TensorRT engine 빌드
`get_engine`은 **로컬/Drive 캐시 → GitHub Release → 직접 빌드** 순서로 engine을 찾습니다.
파일명에 *ONNX 및 빌드 옵션 해시 + GPU 아키텍처 + TRT 버전*이 들어 있어서, 조건이 다르면 자동으로 다시 빌드합니다.
정밀도는 ONNX dtype을 따릅니다(strongly typed). 빌드 과정 자체를 공부할 때는 `force_rebuild=True`를 쓰거나 `trt_utils.build_engine`을 직접 보세요.

In [8]:
from vopt import trt_utils

engine_path = trt_utils.get_engine("resnet18", onnx_path)
trt_model = trt_utils.TRTModule(engine_path)
print("inputs :", trt_model.input_names)
print("outputs:", trt_model.output_names)

fp16_engine_path = trt_utils.get_engine("resnet18_fp16", fp16_onnx_path)
fp16_trt_model = trt_utils.TRTModule(fp16_engine_path)
print("inputs :", fp16_trt_model.input_names)
print("outputs:", fp16_trt_model.output_names)

[cache] not in release: resnet18-2ee96b71-tesla-t4-sm75-trt11.4.0.106.engine
[cache] building: resnet18-2ee96b71-tesla-t4-sm75-trt11.4.0.106.engine
[cache] saved: /content/vopt-cache/resnet18-2ee96b71-tesla-t4-sm75-trt11.4.0.106.engine
inputs : ['input']
outputs: ['logits']
[cache] hit: /content/vopt-cache/resnet18_fp16-34c338eb-tesla-t4-sm75-trt11.4.0.106.engine
inputs : ['input']
outputs: ['logits']


## 3. 정확도 & 속도 비교

In [4]:
model_autocast = copy.deepcopy(model)

def run_autocast(inp):
  with torch.amp.autocast("cuda", dtype=torch.float16):
    return model_autocast(inp)

model_compiled = torch.compile(copy.deepcopy(model))
model_compiled_fp16 = torch.compile(copy.deepcopy(model_fp16))

In [9]:
from vopt.bench import benchmark, compare

compare({
    "pytorch fp32": benchmark(lambda: model(x)),
    "pytorch fp16": benchmark(lambda: model_fp16(x_fp16)),
    "pytorch autocast": benchmark(lambda: run_autocast(x)),
    "pytorch compiled fp32": benchmark(lambda: model_compiled(x)),
    "pytorch compiled fp16": benchmark(lambda: model_compiled_fp16(x_fp16)),
    "tensorrt fp32": benchmark(lambda: trt_model(x)),
    "tensorrt fp16": benchmark(lambda: fp16_trt_model(x_fp16)),
})

name                  mean(ms)       p50       p90   speedup
pytorch fp32             9.900     9.905    10.076     1.00x
pytorch fp16             4.440     4.329     4.810     2.23x
pytorch autocast         4.468     4.429     4.504     2.22x
pytorch compiled fp32     9.672     9.713     9.775     1.02x
pytorch compiled fp16     2.687     2.671     2.760     3.68x
tensorrt fp32            7.568     7.565     7.674     1.31x
tensorrt fp16            0.803     0.782     0.883    12.33x


## 4. (선택) 빌드한 engine 공유하기
다음 study 때나 블로그 독자가 빌드를 건너뛸 수 있도록 GitHub Release에 올립니다.
1. 명령 팔레트 → `Colab: Download...`로 캐시 디렉토리의 `*.engine` 파일을 로컬 `artifacts/`에 받습니다.
2. 로컬에서 `scripts/publish_artifact.sh artifacts/*.engine`을 실행합니다.

engine은 **같은 GPU 아키텍처와 같은 TRT 버전**에서만 재사용할 수 있습니다(예: T4로 빌드한 engine은 L4에서 사용 불가).